# **Tech Challenge - Fase 2**
## Pipeline Híbrido para Análise da Alfabetização no Brasil 📚
### Esta parte do código se refere à segmentação em CLOUD para testes antes de subir ao AWS

Ana Caroline Gonçalves Lima, RM - 373735


In [108]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# basedosdados se refere a base que o Governo Brasileiro disponíbiliza para análises
# pyarrow para salvar em PARQUET

!pip install basedosdados pyarrow --quiet

In [109]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import basedosdados as bd
import pandas as pd
import hashlib
import logging
from pathlib import Path

from datetime import datetime, timezone

In [110]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
PROJECT_ID = "tech-challenge-fase-2-502101"

DATASET = "br_inep_avaliacao_alfabetizacao"

TABELAS = [
    "uf",
    "meta_alfabetizacao_brasil",
    "meta_alfabetizacao_uf",
    "meta_alfabetizacao_municipio",
    "municipio",
    "alunos"
]

BUCKET = "fiap-alfabetizacao-ana-707472259268-us-east-1-an"

INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

CAMADA_BRONZE = "bronze"

In [111]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)

log = logging.getLogger(__name__)

In [112]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA BRONZE")
log.info(f"Projeto GCP : {PROJECT_ID}")
log.info(f"Dataset     : {DATASET}")
log.info(f"Bucket S3   : {BUCKET}")
log.info("~" * 35)

In [113]:
QUERIES = {

    "uf": """
    WITH
dicionario_serie AS (
    SELECT
        chave AS chave_serie,
        valor AS descricao_serie
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'serie'
        AND id_tabela = 'uf'
),
dicionario_rede AS (
    SELECT
        chave AS chave_rede,
        valor AS descricao_rede
    FROM `basedosdados.br_inep_avaliacao_alfabetizacao.dicionario`
    WHERE
        TRUE
        AND nome_coluna = 'rede'
        AND id_tabela = 'uf'
)
SELECT
    dados.ano as ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    descricao_serie AS serie,
    descricao_rede AS rede,
    dados.taxa_alfabetizacao as taxa_alfabetizacao,
    dados.media_portugues as media_portugues,
    dados.proporcao_aluno_nivel_0 as proporcao_aluno_nivel_0,
    dados.proporcao_aluno_nivel_1 as proporcao_aluno_nivel_1,
    dados.proporcao_aluno_nivel_2 as proporcao_aluno_nivel_2,
    dados.proporcao_aluno_nivel_3 as proporcao_aluno_nivel_3,
    dados.proporcao_aluno_nivel_4 as proporcao_aluno_nivel_4,
    dados.proporcao_aluno_nivel_5 as proporcao_aluno_nivel_5,
    dados.proporcao_aluno_nivel_6 as proporcao_aluno_nivel_6,
    dados.proporcao_aluno_nivel_7 as proporcao_aluno_nivel_7,
    dados.proporcao_aluno_nivel_8 as proporcao_aluno_nivel_8
FROM `basedosdados.br_inep_avaliacao_alfabetizacao.uf` AS dados
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
LEFT JOIN `dicionario_serie`
    ON dados.serie = chave_serie
LEFT JOIN `dicionario_rede`
    ON dados.rede = chave_rede
    """

}

In [114]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE LEITURA DAS QUERIES
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_base_dados(query):

    df = bd.read_sql(
        query=query,
        billing_project_id=PROJECT_ID
    )

    return df

In [115]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# EXIBINDO CABEÇALHO DO DATASET
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
ler_base_dados

<function __main__.ler_base_dados(query)>

In [116]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSTRUINDO A CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def construir_bronze(df, dataset, tabela):

    log.info("Adicionando metadados da camada Bronze")

    df = df.copy()

    df["_ingestion_timestamp"] = INGESTION_TS
    df["_ingestion_date"] = INGESTION_DATE
    df["_source_dataset"] = dataset
    df["_source_table"] = tabela

    df["_record_hash"] = (
        df.astype(str)
          .apply(lambda row: hashlib.md5("".join(row).encode()).hexdigest(), axis=1)
    )

    log.info(f"{len(df)} registros preparados para camada Bronze")

    return df

In [117]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE QUALIDADE
# ~~~~~~~~~~~~~~~~~~~~~~~~

CHECKS = {

    "uf": [
        {"tipo": "min_count", "valor": 1, "critico": True},
        {"tipo": "not_null", "coluna": "ano", "critico": True},
        {"tipo": "not_null", "coluna": "sigla_uf", "critico": True},
        {"tipo": "not_null", "coluna": "taxa_alfabetizacao", "critico": True},
    ],

    "municipio": [
        # vamos preencher depois
    ],

    "alunos": [
        # vamos preencher depois
    ],

    "meta_alfabetizacao_uf": [
        # vamos preencher depois
    ],

    "meta_alfabetizacao_municipio": [
        # vamos preencher depois
    ],

    "meta_alfabetizacao_brasil": [
        # vamos preencher depois
    ]
}

In [118]:
# ~~~~~~~~~~~~~~
# DATA QUALITY
# ~~~~~~~~~~~~~~

def checar_qualidade(df, checks):

    log.info("Iniciando verificações de qualidade")

    for check in checks:

        if check["tipo"] == "min_count":

            quantidade = len(df)

            if quantidade < check["valor"]:
                raise Exception(
                    f"Falha: quantidade de registros ({quantidade}) "
                    f"menor que o mínimo esperado ({check['valor']})"
                )

            log.info(f"OK - Quantidade de registros: {quantidade}")

        elif check["tipo"] == "not_null":

            coluna = check["coluna"]

            nulos = df[coluna].isnull().sum()

            if nulos > 0:
                raise Exception(
                    f"Falha: coluna '{coluna}' possui {nulos} valores nulos."
                )

            log.info(f"OK - Coluna '{coluna}' sem valores nulos")

    log.info("Todas as verificações passaram com sucesso.")

In [119]:
# ~~~~~~~~~~~~~~~~~~~~
# SALVAR CAMADA BRONZE
# ~~~~~~~~~~~~~~~~~~~~

def salvar_bronze(df, tabela):

    pasta = Path("bronze")
    pasta.mkdir(exist_ok=True)

    arquivo = pasta / f"{tabela}.parquet"

    log.info(f"Salvando arquivo: {arquivo}")

    df.to_parquet(
        arquivo,
        index=False,
        engine="pyarrow"
    )

    log.info("Arquivo Parquet criado com sucesso.")

    return arquivo

In [120]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO PARA CAMADA BRONZE PARA REUTILIZAR EM VÁRIAS TABELAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def executar_bronze():
    for tabela, query in QUERIES.items():
      log.info("~" * 35)
      log.info(f"Iniciando execução da camada Bronze para '{tabela}'");
      log.info("~" * 35)

      # leitura da base dos dados
      df = ler_base_dados(query)

      # construindo a bronze com os metadados
      df_bronze = construir_bronze(df, DATASET, tabela)

      # exibe as primeiras linhas, somente usado no colab
      print(f"\nPrévia da tabela: {tabela}")
      display(df_bronze.head())

      # checks de integridades e tipos
      checks = CHECKS.get(tabela, [])

      # data quality
      if checks:
         checar_qualidade(df_bronze, checks)

      # salvando
      salvar_bronze(df_bronze, tabela)

log.info("Camada Bronze concluída com sucesso!")

In [121]:
executar_bronze()

Downloading: 100%|██████████|

Prévia da tabela: uf


,ano,sigla_uf,sigla_uf_nome,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,...,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingestion_timestamp,_ingestion_date,_source_dataset,_source_table,_record_hash
0,2023,AM,Amazonas,2° ano do Ensino Fundamental,Municipal,49.20,733.6637,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_164132,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,8c56e13dcac49679a21f8b8c60a881f3
1,2023,PB,Paraíba,2° ano do Ensino Fundamental,Estadual,55.23,744.8152,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_164132,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,ebfd4bed2a590950a02868dc612fd30f
2,2023,PR,Paraná,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),73.12,757.2146,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_164132,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,f3ede63a977ccaaa4f72cb23bbd9a1af
3,2023,AP,Amapá,2° ano do Ensino Fundamental,Municipal,41.87,732.7858,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_164132,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,90d888ff48fadb78920f4691f6fe4632
4,2023,PE,Pernambuco,2° ano do Ensino Fundamental,Pública (Estadual e Municipal),58.95,747.4522,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,20260712_164132,2026-07-12,br_inep_avaliacao_alfabetizacao,uf,5a263841c8477f9841709fc233c02776
